# 06B — Final Metric Evaluation — Fragment-Shot Prompting (Exact Locked) — FINAL V1

Notebook ini dijalankan **setelah tahap 05 S3E + MBR selesai** untuk skenario **Fragment-Shot Prompting (Exact Locked)**.

Pada tahap ini reference Bahasa Indonesia **baru boleh dibaca**, karena generasi kandidat, S3E, dan MBR sudah selesai.

## Evaluasi yang dihitung

- **chrF++** sebagai metrik utama, menggunakan SacreBLEU `CHRF(word_order=2)`
- **BLEU** sebagai metrik pendukung, menggunakan SacreBLEU `BLEU(tokenize="13a")`
- nilai **S3E** dari hasil 05 dipertahankan untuk analisis kestabilan
- detail skor per kalimat disimpan untuk analisis lanjutan

Notebook tidak menjalankan Llama dan tidak mengubah hasil MBR.

## 1. Dependency audit

In [1]:
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

EXPECTED = {
    "sacrebleu": "2.5.1",
}

errors = []
for package_name, expected in EXPECTED.items():
    try:
        actual = version(package_name)
    except PackageNotFoundError:
        errors.append(
            f"{package_name}: NOT INSTALLED. "
            f"Install sekali di terminal: "
            f"/venv/main/bin/python -m pip install '{package_name}=={expected}'"
        )
        continue

    ok = Version(actual) == Version(expected)
    print(
        f"{package_name:18} {actual:12} "
        f"{'OK' if ok else 'MISMATCH'}"
    )
    if not ok:
        errors.append(
            f"{package_name}: actual={actual}, expected={expected}"
        )

if errors:
    raise RuntimeError(
        "Dependency audit gagal:\n- "
        + "\n- ".join(errors)
    )

print("\nDEPENDENCY AUDIT: PASS")

sacrebleu          2.5.1        OK

DEPENDENCY AUDIT: PASS


## 2. Environment

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

SCENARIO_SLUG = "fsp_locked"
SCENARIO_LABEL = "Fragment-Shot Prompting (Exact Locked)"

SELECTION_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "final_selection"
    / SCENARIO_SLUG
)

TEST_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "test.csv"
)

EVAL_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "final_evaluation"
    / SCENARIO_SLUG
)

EXPECTED_TEST_ROWS = 53

if not SELECTION_ROOT.exists():
    raise FileNotFoundError(
        f"Folder final selection tidak ditemukan: {SELECTION_ROOT}"
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Dataset test tidak ditemukan: {TEST_PATH}"
    )

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("SCENARIO       :", SCENARIO_LABEL)
print("SELECTION_ROOT :", SELECTION_ROOT)
print("TEST_PATH      :", TEST_PATH)

PROJECT_ROOT   : /workspace/10122233_CikitaMeydilvira_Skripsi
SCENARIO       : Fragment-Shot Prompting (Exact Locked)
SELECTION_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked
TEST_PATH      : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset/test.csv


## 3. Temukan output 05 yang COMPLETE

In [3]:
def read_json(path: Path):
    return json.loads(
        path.read_text(encoding="utf-8")
    )

candidate_runs = sorted(
    SELECTION_ROOT.glob("run_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

valid_runs = []

for run_dir in candidate_runs:
    selected_path = (
        run_dir / "mbr_selected.csv"
    )
    summary_path = (
        run_dir / "s3e_mbr_summary.json"
    )
    manifest_path = (
        run_dir / "processing_manifest.json"
    )

    if not (
        selected_path.exists()
        and summary_path.exists()
        and manifest_path.exists()
    ):
        continue

    try:
        selected_try = pd.read_csv(
            selected_path,
            dtype={"row_id": str},
            keep_default_na=False,
            encoding="utf-8-sig",
        )

        if len(selected_try) != 53:
            raise RuntimeError(
                f"mbr_selected rows={len(selected_try)}"
            )

        required = {
            "row_id",
            "source_text",
            "selected_output",
            "s3e_value",
            "selection_status",
        }
        missing = required - set(selected_try.columns)
        if missing:
            raise RuntimeError(
                f"Kolom kurang: {sorted(missing)}"
            )

        if selected_try["row_id"].duplicated().any():
            raise RuntimeError(
                "row_id duplikat."
            )

        if (
            selected_try["selected_output"]
            .astype(str)
            .str.strip()
            .eq("")
            .any()
        ):
            raise RuntimeError(
                "Ada selected_output kosong."
            )

        manifest = read_json(manifest_path)
        if manifest.get("reference_used") is not False:
            raise RuntimeError(
                "processing_manifest tidak membuktikan reference_used=False."
            )

        summary = read_json(summary_path)
        if int(summary.get("test_groups", -1)) != 53:
            raise RuntimeError(
                "S3E summary test_groups bukan 53."
            )

        valid_runs.append(
            (
                run_dir,
                selected_path,
                summary_path,
                manifest_path,
                selected_try,
                summary,
                manifest,
            )
        )

    except Exception as exc:
        print(
            "Skip:",
            run_dir,
            "|",
            type(exc).__name__,
            str(exc)[:180],
        )

if not valid_runs:
    raise FileNotFoundError(
        "Tidak ditemukan output 05 yang COMPLETE dan lolos audit."
    )

(
    INPUT_RUN_DIR,
    SELECTED_PATH,
    S3E_SUMMARY_PATH,
    S3E_MANIFEST_PATH,
    selected_df,
    s3e_summary,
    s3e_manifest,
) = valid_runs[0]

INPUT_RUN_NAME = INPUT_RUN_DIR.name
OUTPUT_DIR = (
    EVAL_ROOT / INPUT_RUN_NAME
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("INPUT RUN :", INPUT_RUN_NAME)
print("Selected  :", SELECTED_PATH)
print("Rows      :", len(selected_df))
print("Output    :", OUTPUT_DIR)

display(selected_df.head(8))

INPUT RUN : run_14c344bdb72c6065
Selected  : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked/run_14c344bdb72c6065/mbr_selected.csv
Rows      : 53
Output    : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/fsp_locked/run_14c344bdb72c6065


,scenario,row_id,conversation_no,turn_order,source_text,selected_candidate_index,selected_candidate,selected_output,selected_utility,s3e_value,valid_candidate_count,selection_status,coverage,fragment_count,example_count,max_fragment_len,k_per_fragment
0,fsp_locked,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,1,C1,Pernahkah ada seekor kambing yang berlari di k...,0.558616,7.408302e-01,5,mbr_selected,0.600000,3,7,7,6
1,fsp_locked,C03_T02_B1,3,2,"Ieq, penduanmoq.",2,C2,"Ya, sudah dipelajari.",0.827185,2.735338e-01,5,mbr_selected,0.500000,1,6,7,6
2,fsp_locked,C03_T03_A2,3,3,Piranra tacappaq male tama Panyurak?,5,C5,Kapan baru pergi ke Panyurak?,0.641883,5.086425e-01,5,mbr_selected,0.400000,2,6,7,6
3,fsp_locked,C03_T04_B2,3,4,Ia tonna bulan dua tanggalaq duangpulo annan.,1,C1,Bulan dua tanggal dua puluh.,1.000000,2.717683e-09,5,mbr_selected,1.000000,5,12,7,6
4,fsp_locked,C03_T05_A3,3,5,Mangngaparikiq male tama?,2,C2,Apa yang dikerjakan di rumah? Apa yang dikerja...,0.558573,8.355223e-01,5,mbr_selected,0.666667,2,8,7,6
5,fsp_locked,C03_T06_B3,3,6,Malenaq ngkita bola loqbang.,2,C2,Saya pergi ke pasar.,0.850877,3.107005e-01,5,mbr_selected,0.500000,2,9,7,6
6,fsp_locked,C03_T07_A4,3,7,Ciapari naloqbang joq bola?,3,C3,Mengapa rumah itu runtuh?,0.834481,2.365841e-01,5,mbr_selected,0.750000,3,15,7,6
7,fsp_locked,C03_T08_B4,3,8,"Taqdei puangna, sanga torro jiongri Pinrang.",3,C3,"Hari ini sore, karena kamu tinggal di Pinrang.",0.531342,9.829432e-01,5,mbr_selected,0.333333,2,9,7,6


## 4. Baca reference test dan align berdasarkan row_id

In [4]:
TEST_COLUMNS = [
    "row_id",
    "conversation_no",
    "turn_order",
    "duri_text",
    "indonesia_text",
]

test_df = pd.read_csv(
    TEST_PATH,
    usecols=TEST_COLUMNS,
    dtype={"row_id": str},
    keep_default_na=False,
    encoding="utf-8-sig",
)

if len(test_df) != EXPECTED_TEST_ROWS:
    raise RuntimeError(
        f"Test rows {len(test_df)} != 53."
    )

if test_df["row_id"].duplicated().any():
    raise RuntimeError(
        "row_id test duplikat."
    )

if (
    test_df["indonesia_text"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise RuntimeError(
        "Reference Indonesia kosong."
    )

merged = selected_df.merge(
    test_df,
    on="row_id",
    how="inner",
    validate="one_to_one",
    suffixes=("_selection", "_test"),
)

if len(merged) != 53:
    raise RuntimeError(
        f"Alignment selection-reference {len(merged)} != 53."
    )

def norm_text(x):
    return " ".join(
        str(x).strip().split()
    )

source_match = (
    merged["source_text"]
    .map(norm_text)
    ==
    merged["duri_text"]
    .map(norm_text)
)

if not bool(source_match.all()):
    bad = merged.loc[
        ~source_match,
        ["row_id", "source_text", "duri_text"],
    ]
    display(bad.head(10))
    raise RuntimeError(
        "Source text dari output 05 tidak sama dengan duri_text test."
    )

reference_payload = "\n".join(
    f"{r.row_id}\t{r.indonesia_text}"
    for r in merged.itertuples(index=False)
).encode("utf-8")

REFERENCE_FINGERPRINT = hashlib.sha256(
    reference_payload
).hexdigest()

print("Aligned rows          :", len(merged))
print("Reference read stage  : EVALUATION ONLY")
print("Reference fingerprint :", REFERENCE_FINGERPRINT)

display(
    merged[
        [
            "row_id",
            "source_text",
            "selected_output",
            "indonesia_text",
            "s3e_value",
        ]
    ].head(8)
)

Aligned rows          : 53
Reference read stage  : EVALUATION ONLY
Reference fingerprint : f8b668d87795c26dff25017c1ead195394cff5bfc55f5b75522693bcc667c289


,row_id,source_text,selected_output,indonesia_text,s3e_value
0,C03_T01_A1,Denmoraka naden tamale tama Panyurak?,Pernahkah ada seekor kambing yang berlari di k...,Apakah Anda sudah pernah ke Panyurak?,7.408302e-01
1,C03_T02_B1,"Ieq, penduanmoq.","Ya, sudah dipelajari.","Ya, sudah dua kali.",2.735338e-01
2,C03_T03_A2,Piranra tacappaq male tama Panyurak?,Kapan baru pergi ke Panyurak?,Kapan Anda terakhir pergi ke Panyurak?,5.086425e-01
3,C03_T04_B2,Ia tonna bulan dua tanggalaq duangpulo annan.,Bulan dua tanggal dua puluh.,Pada bulan dua tanggal dua puluh enam.,2.717683e-09
4,C03_T05_A3,Mangngaparikiq male tama?,Apa yang dikerjakan di rumah? Apa yang dikerja...,Ada perlu apa Anda ke sana?,8.355223e-01
5,C03_T06_B3,Malenaq ngkita bola loqbang.,Saya pergi ke pasar.,Saya pergi melihat rumah kosong.,3.107005e-01
6,C03_T07_A4,Ciapari naloqbang joq bola?,Mengapa rumah itu runtuh?,Mengapa rumah itu kosong?,2.365841e-01
7,C03_T08_B4,"Taqdei puangna, sanga torro jiongri Pinrang.","Hari ini sore, karena kamu tinggal di Pinrang.","Yang punya rumah itu tidak ada, karena dia tin...",9.829432e-01


## 5. Hitung chrF++ dan BLEU

In [5]:
bleu_corpus = BLEU(
    tokenize="13a",
)

bleu_sentence = BLEU(
    tokenize="13a",
    effective_order=True,
)

chrfpp = CHRF(
    word_order=2,
)

hypotheses = (
    merged["selected_output"]
    .astype(str)
    .tolist()
)

references = (
    merged["indonesia_text"]
    .astype(str)
    .tolist()
)

corpus_bleu = float(
    bleu_corpus.corpus_score(
        hypotheses,
        [references],
    ).score
)

corpus_chrfpp = float(
    chrfpp.corpus_score(
        hypotheses,
        [references],
    ).score
)

sentence_bleu_scores = []
sentence_chrfpp_scores = []

for hyp, ref in zip(
    hypotheses,
    references,
):
    sentence_bleu_scores.append(
        float(
            bleu_sentence.sentence_score(
                hyp,
                [ref],
            ).score
        )
    )
    sentence_chrfpp_scores.append(
        float(
            chrfpp.sentence_score(
                hyp,
                [ref],
            ).score
        )
    )

details = merged[
    [
        "row_id",
        "conversation_no_test",
        "turn_order_test",
        "duri_text",
        "indonesia_text",
        "selected_candidate_index",
        "selected_candidate",
        "selected_output",
        "selected_utility",
        "s3e_value",
        "valid_candidate_count",
        "selection_status",
    ]
].copy()

details = details.rename(
    columns={
        "conversation_no_test": "conversation_no",
        "turn_order_test": "turn_order",
        "duri_text": "source_text",
        "indonesia_text": "reference_text",
    }
)

details["sentence_bleu_13a"] = (
    sentence_bleu_scores
)
details["sentence_chrfpp"] = (
    sentence_chrfpp_scores
)

print("=" * 70)
print(SCENARIO_LABEL)
print("=" * 70)
print(f"Corpus chrF++ : {corpus_chrfpp:.6f}")
print(f"Corpus BLEU   : {corpus_bleu:.6f}")
print(
    f"Mean S3E     : "
    f"{pd.to_numeric(details['s3e_value'], errors='coerce').mean():.6f}"
)

display(details.head(10))

Fragment-Shot Prompting (Exact Locked)
Corpus chrF++ : 31.839264
Corpus BLEU   : 10.565661
Mean S3E     : 0.691243


,row_id,conversation_no,turn_order,source_text,reference_text,selected_candidate_index,selected_candidate,selected_output,selected_utility,s3e_value,valid_candidate_count,selection_status,sentence_bleu_13a,sentence_chrfpp
0,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,Apakah Anda sudah pernah ke Panyurak?,1,C1,Pernahkah ada seekor kambing yang berlari di k...,0.558616,7.408302e-01,5,mbr_selected,7.495553,32.539646
1,C03_T02_B1,3,2,"Ieq, penduanmoq.","Ya, sudah dua kali.",2,C2,"Ya, sudah dipelajari.",0.827185,2.735338e-01,5,mbr_selected,34.983301,52.633784
2,C03_T03_A2,3,3,Piranra tacappaq male tama Panyurak?,Kapan Anda terakhir pergi ke Panyurak?,5,C5,Kapan baru pergi ke Panyurak?,0.641883,5.086425e-01,5,mbr_selected,45.480190,57.625386
3,C03_T04_B2,3,4,Ia tonna bulan dua tanggalaq duangpulo annan.,Pada bulan dua tanggal dua puluh enam.,1,C1,Bulan dua tanggal dua puluh.,1.000000,2.717683e-09,5,mbr_selected,38.498150,66.655315
4,C03_T05_A3,3,5,Mangngaparikiq male tama?,Ada perlu apa Anda ke sana?,2,C2,Apa yang dikerjakan di rumah? Apa yang dikerja...,0.558573,8.355223e-01,5,mbr_selected,3.386499,13.466782
5,C03_T06_B3,3,6,Malenaq ngkita bola loqbang.,Saya pergi melihat rumah kosong.,2,C2,Saya pergi ke pasar.,0.850877,3.107005e-01,5,mbr_selected,19.357693,33.197576
6,C03_T07_A4,3,7,Ciapari naloqbang joq bola?,Mengapa rumah itu kosong?,3,C3,Mengapa rumah itu runtuh?,0.834481,2.365841e-01,5,mbr_selected,42.728701,65.253991
7,C03_T08_B4,3,8,"Taqdei puangna, sanga torro jiongri Pinrang.","Yang punya rumah itu tidak ada, karena dia tin...",3,C3,"Hari ini sore, karena kamu tinggal di Pinrang.",0.531342,9.829432e-01,5,mbr_selected,23.142716,44.751793
8,C03_T09_A5,3,9,Makassingraka joq bola?,Apakah rumah itu bagus?,5,C5,Siapakah orang-orang di rumah?,0.707455,5.661558e-01,5,mbr_selected,12.703319,26.177118
9,C03_T10_B5,3,10,"Makassing na matonggo, pa masussa to wai.","Bagus dan besar, tetapi airnya agak sulit.",3,C3,"Sudah tiba waktunya, mana yang lebih sulit?",0.367758,1.328715e+00,5,mbr_selected,5.669791,17.555198


## 6. Simpan hasil evaluasi

In [6]:
DETAIL_PATH = (
    OUTPUT_DIR
    / "final_metric_details.csv"
)
SUMMARY_PATH = (
    OUTPUT_DIR
    / "final_metric_summary.json"
)
MANIFEST_PATH = (
    OUTPUT_DIR
    / "evaluation_manifest.json"
)

details.to_csv(
    DETAIL_PATH,
    index=False,
    encoding="utf-8-sig",
)

s3e_numeric = pd.to_numeric(
    details["s3e_value"],
    errors="coerce",
)

summary = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "scenario":
        SCENARIO_SLUG,
    "scenario_label":
        SCENARIO_LABEL,
    "input_run_name":
        INPUT_RUN_NAME,
    "test_rows":
        int(len(details)),
    "primary_metric":
        "chrF++",
    "corpus_chrfpp":
        corpus_chrfpp,
    "corpus_bleu_13a":
        corpus_bleu,
    "mean_s3e":
        float(s3e_numeric.mean()),
    "median_s3e":
        float(s3e_numeric.median()),
    "s3e_available_rows":
        int(s3e_numeric.notna().sum()),
    "bleu_tokenizer":
        "13a",
    "chrf_word_order":
        2,
    "reference_used":
        True,
    "reference_stage":
        "final_evaluation_only",
    "reference_fingerprint":
        REFERENCE_FINGERPRINT,
    "selected_output_path":
        str(SELECTED_PATH),
    "details_path":
        str(DETAIL_PATH),
}

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

manifest = {
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "scenario":
        SCENARIO_SLUG,
    "input_selected_sha256":
        hashlib.sha256(
            SELECTED_PATH.read_bytes()
        ).hexdigest(),
    "test_csv_sha256":
        hashlib.sha256(
            TEST_PATH.read_bytes()
        ).hexdigest(),
    "sacrebleu_version":
        sacrebleu.__version__,
    "python":
        platform.python_version(),
    "metric_protocol": {
        "bleu": {
            "library": "sacrebleu",
            "tokenize": "13a",
            "level": "corpus",
        },
        "chrfpp": {
            "library": "sacrebleu",
            "word_order": 2,
            "level": "corpus",
        },
    },
    "reference_used":
        True,
    "reference_stage":
        "final_evaluation_only",
    "outputs": {
        "details":
            str(DETAIL_PATH),
        "summary":
            str(SUMMARY_PATH),
    },
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("DETAILS :", DETAIL_PATH)
print("SUMMARY :", SUMMARY_PATH)
print("MANIFEST:", MANIFEST_PATH)

DETAILS : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/fsp_locked/run_14c344bdb72c6065/final_metric_details.csv
SUMMARY : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/fsp_locked/run_14c344bdb72c6065/final_metric_summary.json
MANIFEST: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_evaluation/fsp_locked/run_14c344bdb72c6065/evaluation_manifest.json


## 7. Final audit

In [7]:
checks = {
    "rows_53":
        len(details) == 53,
    "reference_nonempty":
        bool(
            details["reference_text"]
            .astype(str)
            .str.strip()
            .ne("")
            .all()
        ),
    "hypothesis_nonempty":
        bool(
            details["selected_output"]
            .astype(str)
            .str.strip()
            .ne("")
            .all()
        ),
    "details_exists":
        DETAIL_PATH.exists(),
    "summary_exists":
        SUMMARY_PATH.exists(),
    "manifest_exists":
        MANIFEST_PATH.exists(),
    "finite_chrfpp":
        np.isfinite(corpus_chrfpp),
    "finite_bleu":
        np.isfinite(corpus_bleu),
}

failed = [
    k for k, v in checks.items()
    if not bool(v)
]

display(
    pd.DataFrame(
        [
            {"check": k, "ok": bool(v)}
            for k, v in checks.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL EVALUATION AUDIT GAGAL: {failed}"
    )

print("\n" + "=" * 74)
print(
    f"{SCENARIO_LABEL} — FINAL METRIC EVALUATION: COMPLETE"
)
print("=" * 74)
print(f"chrF++ : {corpus_chrfpp:.6f}")
print(f"BLEU   : {corpus_bleu:.6f}")
print(
    "\nSetelah 06A, 06B, dan 06C selesai, "
    "jalankan 06D untuk perbandingan dan paired bootstrap."
)

,check,ok
0,rows_53,True
1,reference_nonempty,True
2,hypothesis_nonempty,True
3,details_exists,True
4,summary_exists,True
5,manifest_exists,True
6,finite_chrfpp,True
7,finite_bleu,True



Fragment-Shot Prompting (Exact Locked) — FINAL METRIC EVALUATION: COMPLETE
chrF++ : 31.839264
BLEU   : 10.565661

Setelah 06A, 06B, dan 06C selesai, jalankan 06D untuk perbandingan dan paired bootstrap.
